In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import time
import keras
import numpy as np
# Necesario para TextVectorization y tf.data.
import tensorflow as tf
from models.training import compile_model, get_callbacks
from config.settings import Settings
from features.embeddings import load_gensim_embeddings
from datasets.dataset import create_dataset
from features.vectorizer import build_vectorizer
from datasets.loader import load_splits, save_json
from models.siamese_lstm import SiameseLSTM
from datasets.paths import ProjectPaths


In [2]:
print(keras.config.backend())


torch


In [3]:
SEED = 42
np.random.seed(SEED)


In [ ]:
settings = Settings()
print(settings)


embed_dim=400 hidden_dim=64 batch_size=64 mlp_dropout=0.4 lstm_dropout=0.3 pooling='mean' similarity='cosine' mlp_layers=[16] bidirectional=True concat_features=['diff'] epochs=20 siamese_name='bilstm_mean_cosine'


In [ ]:
paths = ProjectPaths(siamese_name=settings.siamese_name)


In [ ]:
if settings.augmented_data:
	max_len = 27
	train_dir = paths.augmented_dir
else:
	max_len = 26
	train_dir = paths.processed_dir


In [ ]:
splits = {
	"train": train_dir,
	"dev": paths.processed_dir,
    "test": paths.processed_dir
}

datasets = load_splits(splits)

train_df = datasets["train"]
dev_df = datasets["dev"]
test_df = datasets["test"]


In [7]:
print("Train length:", len(train_df))
print("Dev length:", len(dev_df))


Train length: 15165
Dev length: 1497


In [ ]:
all_sentences = list(train_df["sentence1"]) + list(train_df["sentence2"])

vectorizer = build_vectorizer(all_sentences, max_len)

vocab = vectorizer.get_vocabulary()
word2idx = {word: idx for idx, word in enumerate(vocab)}
print(f"Vocabulary size: {len(vocab)}")

vectorizer_model = keras.Sequential([vectorizer])
vectorizer_model.save(paths.vectorizer_path)


Vocabulary size: 15886


c:\Users\malos\Documents\GitHub\JustShare\server\.venv\Lib\site-packages\keras\src\saving\saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


In [ ]:
embedding_matrix = load_gensim_embeddings(paths.word2vec_path, word2idx, settings.embed_dim)

np.save(paths.embedding_path, embedding_matrix)


Found 15447/15886 words


In [10]:
train_dataset = create_dataset(train_df, vectorizer, settings.batch_size, shuffle=True)
dev_dataset = create_dataset(dev_df, vectorizer, settings.batch_size)


In [11]:
for (sent1, sent2), y in train_dataset.take(1):
	print("sent1:", sent1.shape)
	print("sent2:", sent2.shape)
	print("y:", y.shape)


sent1: (64, 27)
sent2: (64, 27)
y: (64,)


In [12]:
model = SiameseLSTM(
	vocab_size=len(vocab),
	embedding_dim=settings.embed_dim,
	hidden_dim=settings.hidden_dim,
	mlp_dropout=settings.mlp_dropout,
	lstm_dropout=settings.lstm_dropout,
	embedding_matrix=embedding_matrix,
	pooling=settings.pooling,
	similarity=settings.similarity,
	mlp_layers=settings.mlp_layers,
	bidirectional=settings.bidirectional,
	concat_features=settings.concat_features,
    name=settings.siamese_name
)


In [13]:
if model.mlp:
	model.mlp.summary()


In [14]:
head_model = model.get_head_model()
head_model.summary()


Model: "siamese_head"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 400) │  6,354,400 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast (Cast)         │ (None, None)      │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm              │ (None, None, 128) │    238,080 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, None, 1)   │          0 │ cast[0][0]        │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, None, 128) │          0 │ bilstm[0][0],     │
│                     │                   │            │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum_1 (Sum)         │ (None, 1)         │          0 │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum (Sum)           │ (None, 128)       │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1)         │          0 │ sum_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ true_divide         │ (None, 128)       │          0 │ sum[0][0],        │
│ (TrueDivide)        │                   │            │ add[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,592,480 (25.15 MB)

 Trainable params: 238,080 (930.00 KB)

 Non-trainable params: 6,354,400 (24.24 MB)

In [15]:
dummy_sent1 = tf.zeros((1, max_len), dtype=tf.int32)
dummy_sent2 = tf.zeros((1, max_len), dtype=tf.int32)

model((dummy_sent1, dummy_sent2))

model.summary()


Model: "bilstm_mean_cosine"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 400)      │     6,354,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ (None, None, 128)      │       238,080 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,592,480 (25.15 MB)

 Trainable params: 238,080 (930.00 KB)

 Non-trainable params: 6,354,400 (24.24 MB)

In [ ]:
model = compile_model(model)

callbacks = get_callbacks(paths.siamese_path)


In [ ]:
start_time = time.perf_counter()

history = model.fit(
	train_dataset,
	validation_data=dev_dataset,
	epochs=settings.epochs,
	callbacks=callbacks
)

train_time = time.perf_counter() - start_time

np.save(paths.history_path, history.history)


Epoch 1/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 270s 1s/step - loss: 0.0591 - mae: 0.1999 - rmse: 0.2431 - val_loss: 0.0960 - val_mae: 0.2505 - val_rmse: 0.3099 - learning_rate: 0.0010
Epoch 2/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 289s 1s/step - loss: 0.0442 - mae: 0.1713 - rmse: 0.2102 - val_loss: 0.0783 - val_mae: 0.2235 - val_rmse: 0.2799 - learning_rate: 0.0010
Epoch 3/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 279s 1s/step - loss: 0.0363 - mae: 0.1541 - rmse: 0.1906 - val_loss: 0.0726 - val_mae: 0.2131 - val_rmse: 0.2695 - learning_rate: 0.0010
Epoch 4/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 305s 1s/step - loss: 0.0310 - mae: 0.1424 - rmse: 0.1762 - val_loss: 0.0683 - val_mae: 0.2071 - val_rmse: 0.2614 - learning_rate: 0.0010
Epoch 5/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 301s 1s/step - loss: 0.0273 - mae: 0.1328 - rmse: 0.1651 - val_loss: 0.0665 - val_mae: 0.2041 - val_rmse: 0.2579 - learning_rate: 0.0010
Epoch 6/20
237/237 ━━━━━━━━━━━━━━━━━━━━ 293s 1s/step - loss: 0.0237 - mae: 0.1232 - rmse: 0.1538 - val_loss: 0.0640

In [ ]:
run_config = {
    "sequence_length": max_len,
    "data_augmentation": settings.augmented_data,
    "train_time_s": train_time
}

save_json(run_config, paths.config_path)
